In [10]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface, Probe
from scipy.io import loadmat


probe_data = loadmat("/media/ubuntu/sda/duan/raw_data/chanMap_DCX_5mm.mat")
probe_x = probe_data['xcoords']
probe_y = probe_data['ycoords']

probe_position = pd.DataFrame(probe_x)
probe_position[1] = probe_y

probe = Probe()
probe.set_contacts(positions=probe_position, contact_ids=probe_data['chanMap'][:, 0])

probe_loc = pd.read_csv('/media/ubuntu/sda/duan/raw_data/ch_map_R.csv')
probe.set_device_channel_indices(probe_loc['probeloc'].values)



In [ ]:
recording_raw = se.read_intan(f"/home/ubuntu/Downloads/grid/M190011_250521_141514_merged_130.rhd", stream_id= '0', ignore_integrity_checks=True)

print('read success')

recording_raw = spre.unsigned_to_signed(recording_raw)
recording_recorded = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_recorded, freq=50)
recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

recording_f = recording_f.set_probegroup(probe)
recording_preprocessed = recording_f.save(format="binary")


read success
Use cache_folder=/tmp/spikeinterface_cache/tmp56kt7jau/TPBTUY4H
write_binary_recording 
engine=process - n_jobs=1 - samples_per_chunk=30,000 - chunk_memory=14.65 MiB - total_memory=14.65 MiB - chunk_duration=1.00s


noise_level (no parallelization): 100%|██████████| 20/20 [00:01<00:00, 16.33it/s]
Compute : spike_amplitudes + spike_locations (no parallelization): 100%|██████████| 5262/5262 [01:59<00:00, 44.08it/s]
extract PCs (no parallelization): 100%|██████████| 5262/5262 [1:31:49<00:00,  1.05s/it]  

Run:
phy template-gui  /media/ubuntu/sda/duan/result/phy_folder_for_kilosort/params.py


In [12]:
for rep in range(2, 6):
    output_folder = f'/media/ubuntu/sda/duan/result/rep_{rep}'
    os.makedirs(output_folder, exist_ok=True)

    sorting_kilosort4 = ss.run_sorter(sorter_name="kilosort4", recording=recording_preprocessed, folder=output_folder + "/kilosort4")
    analyzer_kilosort4 = si.create_sorting_analyzer(sorting=sorting_kilosort4, recording=recording_preprocessed, format='binary_folder', folder=output_folder + '/analyzer_kilosort4_binary')

    extensions_to_compute = [
        "random_spikes",
        "waveforms",
        "noise_levels",
        "templates",
        "spike_amplitudes",
        "unit_locations",
        "spike_locations",
        "correlograms",
        "template_similarity"
    ]

    extension_params = {
        "unit_locations": {"method": "center_of_mass"},
        "spike_locations": {"ms_before": 0.1},
        "correlograms": {"bin_ms": 0.1},
        "template_similarity": {"method": "cosine_similarity"}
    }

    analyzer_kilosort4.compute(extensions_to_compute, extension_params=extension_params)

    qm_params = sqm.get_default_qm_params()
    analyzer_kilosort4.compute("quality_metrics", qm_params)

    import spikeinterface.exporters as sexp
    sexp.export_to_phy(analyzer_kilosort4, output_folder + "/phy_folder_for_kilosort", verbose=True)

compute_waveforms (no parallelization): 100%|██████████| 5262/5262 [01:57<00:00, 44.94it/s]
Compute : spike_amplitudes + spike_locations (no parallelization): 100%|██████████| 5262/5262 [02:01<00:00, 43.31it/s]
extract PCs (no parallelization): 100%|██████████| 5262/5262 [1:32:17<00:00,  1.05s/it]  


Run:
phy template-gui  /media/ubuntu/sda/duan/result/rep_2/phy_folder_for_kilosort/params.py


compute_waveforms (no parallelization): 100%|██████████| 5262/5262 [01:53<00:00, 46.46it/s]
Compute : spike_amplitudes + spike_locations (no parallelization): 100%|██████████| 5262/5262 [01:33<00:00, 56.08it/s]
extract PCs (no parallelization): 100%|██████████| 5262/5262 [1:31:06<00:00,  1.04s/it]  


Run:
phy template-gui  /media/ubuntu/sda/duan/result/rep_3/phy_folder_for_kilosort/params.py


compute_waveforms (no parallelization): 100%|██████████| 5262/5262 [03:17<00:00, 26.63it/s] 
Compute : spike_amplitudes + spike_locations (no parallelization): 100%|██████████| 5262/5262 [01:53<00:00, 46.38it/s]
extract PCs (no parallelization): 100%|██████████| 5262/5262 [1:32:26<00:00,  1.05s/it]  


Run:
phy template-gui  /media/ubuntu/sda/duan/result/rep_4/phy_folder_for_kilosort/params.py


compute_waveforms (no parallelization): 100%|██████████| 5262/5262 [02:04<00:00, 42.10it/s]
Compute : spike_amplitudes + spike_locations (no parallelization): 100%|██████████| 5262/5262 [01:36<00:00, 54.66it/s]  
extract PCs (no parallelization): 100%|██████████| 5262/5262 [1:31:18<00:00,  1.04s/it]  

Run:
phy template-gui  /media/ubuntu/sda/duan/result/rep_5/phy_folder_for_kilosort/params.py
